# Nemotron S7 TIES Sign Merge

Mechanism: `ties_trim_elect_sign_merge`.

Hypothesis: reduce destructive interference between several 0.86-class adapter anchors.

This adapter-level operation does not load the 30B base model.

It writes a structurally checked `/kaggle/working/submission.zip` and never submits it.



In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import re
import shutil
import zipfile

import torch
from safetensors.torch import load_file, save_file

CANDIDATE = "nemotron-s7-ties-sign-merge"
METHOD = "ties"
INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
OUT_DIR = WORKING_ROOT / CANDIDATE
ADAPTER_DIR = OUT_DIR / "adapter"
SUBMISSION_ZIP = WORKING_ROOT / "submission.zip"
PREFERRED = ["public_hk_to_kn_lm_head_lam1p0", "public_kn_to_hk_lm_head_lam1p0", "public_hk_to_kn_mamba_lam1p0"]


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_adapters():
    found = []
    for cfg_path in sorted(INPUT_ROOT.rglob("adapter_config.json")):
        model_path = cfg_path.parent / "adapter_model.safetensors"
        if not model_path.exists():
            continue
        cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
        lower = str(cfg_path.parent).lower()
        slug = next((name for name in PREFERRED if name.lower() in lower), cfg_path.parent.name)
        found.append({"slug": slug, "cfg": cfg_path, "model": model_path, "rank": cfg.get("r"), "targets": cfg.get("target_modules")})
    by_slug = {item["slug"]: item for item in found}
    selected = [by_slug[name] for name in PREFERRED if name in by_slug]
    if len(selected) < 3:
        selected = found[:3]
    if len(selected) < 3:
        raise RuntimeError(f"need three compatible adapters, found {len(selected)}")
    ranks = {int(item["rank"]) for item in selected}
    targets = {json.dumps(item["targets"], sort_keys=True) for item in selected}
    if max(ranks) > 32 or len(targets) != 1:
        raise RuntimeError(f"incompatible adapter configs: ranks={ranks}, target_sets={len(targets)}")
    return selected


def layer_number(key):
    match = re.search(r"\.layers\.(\d+)\.", key)
    return int(match.group(1)) if match else -1


def merge_tensor(key, tensors):
    base = tensors[0].float()
    deltas = [tensor.float() - base for tensor in tensors[1:]]
    if METHOD == "ties":
        stacked = torch.stack(deltas)
        threshold = torch.quantile(stacked.abs().reshape(-1), 0.20)
        trimmed = torch.where(stacked.abs() >= threshold, stacked, torch.zeros_like(stacked))
        elected = torch.sign(trimmed.sum(dim=0))
        agrees = torch.sign(trimmed) == elected.unsqueeze(0)
        counts = agrees.sum(dim=0).clamp_min(1)
        merged_delta = torch.where(agrees, trimmed, torch.zeros_like(trimmed)).sum(dim=0) / counts
        return base + 0.55 * merged_delta
    if METHOD == "dare":
        generator = torch.Generator(device="cpu").manual_seed(20260606 + max(0, layer_number(key)))
        kept = []
        density = 0.55
        for delta in deltas:
            mask = torch.rand(delta.shape, generator=generator) < density
            kept.append(torch.where(mask, delta / density, torch.zeros_like(delta)))
        return base + 0.45 * torch.stack(kept).mean(dim=0)
    if METHOD == "layerwise":
        lower = key.lower()
        if "mamba" in lower or "in_proj" in lower or "out_proj" in lower:
            weights = [0.45, 0.35, 0.20]
        elif "expert" in lower or "gate_up_proj" in lower or "down_proj" in lower:
            weights = [0.55, 0.20, 0.25]
        elif "lm_head" in lower:
            weights = [0.35, 0.45, 0.20]
        else:
            weights = [0.50, 0.25, 0.25]
        return sum(weight * tensor.float() for weight, tensor in zip(weights, tensors))
    raise RuntimeError(f"unknown method {METHOD}")


selected = find_adapters()
print("selected_adapters:", [(item["slug"], item["rank"], str(item["model"])) for item in selected])
all_tensors = [load_file(str(item["model"]), device="cpu") for item in selected]
keys = sorted(all_tensors[0])
for item in all_tensors[1:]:
    if sorted(item) != keys:
        raise RuntimeError("adapter tensor keys differ")

merged = {}
for index, key in enumerate(keys):
    values = [item[key] for item in all_tensors]
    shapes = {tuple(value.shape) for value in values}
    if len(shapes) != 1:
        raise RuntimeError(f"shape mismatch for {key}: {shapes}")
    merged[key] = merge_tensor(key, values).to(values[0].dtype)
    if index % 250 == 0:
        print("merged_tensors:", index, "/", len(keys))

OUT_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
cfg_out = ADAPTER_DIR / "adapter_config.json"
model_out = ADAPTER_DIR / "adapter_model.safetensors"
shutil.copyfile(selected[0]["cfg"], cfg_out)
save_file(merged, str(model_out))
if model_out.stat().st_size < 100 * 1024 * 1024:
    raise RuntimeError(
        f"adapter_model.safetensors is unexpectedly small: {model_out.stat().st_size} bytes"
    )
del all_tensors, merged
gc.collect()

if SUBMISSION_ZIP.exists():
    SUBMISSION_ZIP.unlink()
with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_STORED) as zf:
    zf.write(cfg_out, "adapter_config.json")
    zf.write(model_out, "adapter_model.safetensors")
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zf:
    names = sorted(zf.namelist())
if names != ["adapter_config.json", "adapter_model.safetensors"]:
    raise RuntimeError(f"bad zip contents: {names}")
if SUBMISSION_ZIP.stat().st_size < 100 * 1024 * 1024:
    raise RuntimeError(
        f"submission.zip is unexpectedly small: {SUBMISSION_ZIP.stat().st_size} bytes"
    )
report = {
    "candidate": CANDIDATE,
    "method": METHOD,
    "source_adapters": [item["slug"] for item in selected],
    "adapter_model_sha256": sha256_file(model_out),
    "submission_zip_sha256": sha256_file(SUBMISSION_ZIP),
    "submission_zip_size_bytes": SUBMISSION_ZIP.stat().st_size,
    "zip_namelist": names,
    "rank_lte_32": True,
}
(WORKING_ROOT / f"{CANDIDATE}_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))
print("OK: /kaggle/working/submission.zip is ready.")
